In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
# Dataset
# For this homework, we'll use the 2026 Car Fuel Efficiency dataset. Download it from here.

# You can do it with wget:

df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')
#The goal of this homework is to create a regression model for predicting the car fuel efficiency (column 'fuel_efficiency_mpg').

# Preparing the dataset
# Use only the following columns:
cols = ['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']
# EDA
df = df[cols]
base = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]


In [2]:
# Question 1
# There's one column with missing values. What is it?

#'engine_displacement'
#'horsepower' -
#'vehicle_weight'
#'model_year'
df.isnull().sum() # Horsepower

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [3]:
# Question 2
# What's the median (50% percentile) for variable 'horsepower'?

# 204
# 254 - median
# 304
# 354

df['horsepower'].median() # -254

np.float64(254.0)

In [4]:
# Question 3
# We need to deal with missing values for the column from Q1.
# We have two options: fill it with 0 or with the mean of this variable.
# Try both options. For each, train a linear regression model without regularization using the code from the lessons.
# For computing the mean, use the training only!
# Use the validation dataset to evaluate the models and compare the RMSE of each option.
# Round the RMSE scores to 3 decimal digits using round(score, 3). This keeps the imputation difference visible in this release.
# Which option gives better RMSE?
# Options:

# With 0
# With mean
# Both are equally good - Arredondando para 3 casas decimais.

n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

In [5]:
def prepare_X(df):
    df_num = df[base]
    df_num = df_num.fillna(0)
    return df_num.values

In [6]:
def train_linear_regression(X, y):
    one = np.ones(X.shape[0])
    X = np.column_stack([one, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [7]:
def rmse(y, y_pred):
    error = y - y_pred
    return np.sqrt(np.mean(error ** 2))

In [8]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

score = rmse(y_val, y_pred)
print(round(score, 3))
# 0    = 0.07148851150917841
# mean = 0.07136852715046792

0.071


In [9]:
# Question 4
# Now let's train a regularized linear regression.
# For this question, fill the NAs with 0.
# Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].
# Use RMSE to evaluate the model on the validation dataset.
# Round the RMSE scores to 4 decimal digits. This keeps the small but real regularization differences visible instead of turning several choices into a tie.
# Which r gives the best RMSE?
# If multiple options give the same best RMSE, select the smallest r.

# Options:

# 0 - Não acho que esteja correto devido a diferença so aparecer na 6 casa decimal
# 0.01
# 0.1
# 1
# 5
# 10
# 100

In [10]:
def train_linear_regression_r(X, y, r=0.0):
    one = np.ones(X.shape[0])
    X = np.column_stack([one, X]).round(1)

    XTX = X.T.dot(X)

    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [11]:
def train_linear_regression(X, y):
    one = np.ones(X.shape[0])
    X = np.column_stack([one, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

def prepare_X(df, fill_value):
    df_num = df[base]
    df_num = df_num.fillna(fill_value)
    return df_num.values

for r in [0,0.01,0.1,1,5,10,100]:
    X_train = prepare_X(df_train,fill_value=0)
    w0, w = train_linear_regression_r(X_train, y_train, r=r)

    X_val = prepare_X(df_val, fill_value=0)
    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    print(r , score)

0 0.07148851150917841
0.01 0.0714932588415389
0.1 0.07165023237517432
1 0.07274035044832065
5 0.07327911622523631
10 0.0733704123145022
100 0.07345871819236553


In [12]:
# Question 5
# We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
# Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
# For each seed, do the train/validation/test split with 60%/20%/20% distribution.
# Fill the missing values with 0 and train a model without regularization.
# For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
# What's the standard deviation of all the scores? To compute the standard deviation, use np.std.
# Round the result to 3 decimal digits (round(std, 3))
# What's the value of std?

# 0.006
# 0.016
# 0.029
# 0.036
# Note: Standard deviation shows how different the values are. If it's low, then all values are approximately the same. If it's high, the values are different. If standard deviation of scores is low, then our model is stable.

In [13]:
scores = []

for seed in range(10):

    n = len(df)

    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)

    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
    y_val = np.log1p(df_val.fuel_efficiency_mpg.values)

    X_train = prepare_X(df_train, 0)

    w0, w = train_linear_regression(X_train, y_train)

    X_val = prepare_X(df_val, 0)

    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    print(seed, score)

    scores.append(score)

std = np.std(scores)

print("std:", std)
print("rounded:", round(std, 3))

0 0.07222016116435659
1 0.07085480962390645
2 0.07001674410142288
3 0.07160947080489961
4 0.07121934602794797
5 0.07272074319560463
6 0.07363981939829584
7 0.07093670406151228
8 0.07159357901409537
9 0.0716289469263466
std: 0.0009710806116179615
rounded: 0.001


In [14]:
# Question 6
# Split the dataset like previously, use seed 9.
# Combine train and validation datasets.
# Fill the missing values with 0 and train a model with r=0.001.
# What's the RMSE on the test dataset?
# Options:

# 0.236
# 2.236
# 22.10
# 221.0

In [15]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

# 2. Criar os targets
y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

# 3. Juntar train + validation
df_full_train = pd.concat([df_train, df_val])

y_full_train = np.concatenate([y_train, y_val])

# 4. Preparar X do treino completo
#    NaN preenchido com 0
X_full_train = prepare_X(df_full_train, 0)

# 5. Treinar com regularização r = 0.001
w0, w = train_linear_regression_r(
    X_full_train,
    y_full_train,
    r=0.001
)

# 6. Preparar X do teste
X_test = prepare_X(df_test, 0)

# 7. Fazer previsão
y_pred = w0 + X_test.dot(w)

# 8. Calcular RMSE
score = rmse(y_test, y_pred)

print("RMSE:", score)
print("RMSE arredondado:", round(score, 3))

RMSE: 0.07213088684591543
RMSE arredondado: 0.072


In [ ]:
%%sql


In [16]:
print(df.shape)
print(base)
print(y_test[:5])
print(X_test[:5])

(10000, 5)
['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
[3.40119738 3.60277676 3.30321697 3.48431229 3.44680789]
[[2100.  258. 4060. 2011.]
 [2160.  223. 4130. 2011.]
 [2190.    0. 4030. 1975.]
 [2320.  257. 4640. 1997.]
 [2330.  235. 4130. 1988.]]


In [17]:
print(w0)
print(w)

-2.5102733469826877
[-4.40786166e-05 -9.82436337e-06 -1.31826785e-04  3.30418135e-03]


w0 = -2.5102733469826877
w = [-4.40786166e-05 -9.82436337e-06 -1.31826785e-04  3.30418135e-03]
RMSE = 0.07347037390827028
